### Importation 

In [1]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

import sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

pd.set_option('display.max_columns', None)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\eyadh\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\eyadh\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\eyadh\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\eyadh\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\eyadh\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\eyadh\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\eyadh\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\eyadh\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:
data = pd.read_csv("mat_brazil.xls", encoding='latin1')


### Processing

In [ ]:
def traduction(df):
    code = {'SIM':'yes',
            'sim':'yes',
            'nao':'no',
            'nào':'no',
            'NAO':'no',
            'indeterminada':'indeterminate',
            'outros':'other',
            'conservador':'conservative',
            'falecido':'deceased',
            'vivo':'alive',
            'CerebroVascular':'CerebroVascular',
            'identico':0,
            'DISTINTO': 6 ,
            'distinto':'different',
            'haploidentico':3,
            'timo':'thymogllubulin', 
            'tratamento':'treatment',
            'profilatico':'prophylactic',
            'choque séptico':'septic shock',
            'choque septico':'septic shock',
            'choque hemorragico/septico':'hemorrhagic/septic/shock',
            'choque séptico + hemorragico':'septic shock + hemorrhagic shock',
            '-':'no information',
            'doacao':'donation',
            'judical':'judicial',
            'nao recebeu':'did not receive',
            'sem mutação':'no mutation',
            'não identificada':'not identified',
            'nao realizada':'not carried out',
            'ausencia':'absence',
            'dialise' : np.nan,
            }
    
    # Special mappings for specific columns
    final_creatinine_code = code.copy()
    final_creatinine_code['HD'] = np.nan
    final_creatinine_code['DP'] = np.nan
    
    for col in df.select_dtypes('object').columns:
        if col == 'final_creatinine':
            df.loc[:,col] = df[col].replace(final_creatinine_code)
        else:
            df.loc[:,col] = df[col].replace(code)
    
    # Convert final_creatinine to float after replacing
    if 'final_creatinine' in df.columns:
        df['final_creatinine'] = df['final_creatinine'].astype(str).str.replace(',', '.', regex=False).astype(float, errors='ignore')
    
    
    
    return df



def drop_columns(df):
    df = df.drop(columns=['date.tx','cold_ischemia_time' , 'ohter_information.Tx','cause_convertion', 'date.diag' , 'date_eculizumab' , 'time_eculizumabe', 'variant' , 'ADAMST13' , 'Shigatox' , 'biopsy' , 'trat.suport','change.IS' , 'time.trat.ecu' , 'intercorrence.trat' , 'date_event' ,'genetic_description'] , axis = 1)
    return df


def nan_values(df):
    df.loc[df['sex'] == 'M', 'n_pregnant'] = 0
    
    # Fill NaN in numerical columns with their median
    numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
    df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())
    
    # Fill NaN in object columns with their mode (most frequent value)
    object_cols = df.select_dtypes(include=['object']).columns
    for col in object_cols:
        if not df[col].mode().empty:
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)
            
    df.loc[df['death'] == 'no', 'cause.death'] = 'still alive'
    
    df['MM'] = df['MM'].astype(int)
    
    return df


def processing(df):
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    numerical_cols = df.select_dtypes(include='number').columns.tolist()

    transformer = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(sparse_output=False), categorical_cols),
    ('scaler', StandardScaler(), numerical_cols)
    ])

    df = transformer.fit_transform(df)
    
    
    return df 


    
  

def pipeline(df):
    
    df = traduction(df)
    df = drop_columns(df)
    df = nan_values(df)
   
    
    return df

In [ ]:
df_processed = pipeline(data)
df_processed.head(5)
df_processed.dtypes



### Model Training

remarque : TRS 

In [ ]:
from sklearn.model_selection import train_test_split

X = df_processed.drop('rejection', axis = 1)
y = df_processed['rejection']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2,
                                                   random_state = 42, 
                                                   stratify = y
                                                   )

In [ ]:
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(include='number').columns.tolist()

transformer = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_cols),
    ('scaler', StandardScaler(), numerical_cols)
])

# Fit transformer on X_train, then use it for both
X_train = transformer.fit_transform(X_train)
X_test = transformer.transform(X_test)

print(X_train.shape, X_test.shape)

(30, 88) (8, 88)


## Random Forest Classifier 

In [ ]:

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [250],
    'criterion': ["gini", "entropy", "log_loss"],
    'max_depth': [None, 10, 20,30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'bootstrap': [False],
    'class_weight': ["balanced", "balanced_subsample"]
}

grid_search = GridSearchCV(RandomForestClassifier(), param_grid=param_grid, cv=5,scoring='recall_weighted')
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Estimator:", grid_search.best_estimator_)



Best Parameters: {'bootstrap': False, 'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 250}
Best Estimator: RandomForestClassifier(bootstrap=False, class_weight='balanced',
                       criterion='entropy', max_depth=20, min_samples_leaf=2,
                       min_samples_split=5, n_estimators=250)


#### Best Hyperparamaters :

'bootstrap': False

'class_weight': 'balanced'

'criterion': 'log_loss'

'max_depth': 20

'min_samples_leaf': 1

'min_samples_split': 2

'n_estimators': 250

In [ ]:
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
model_grid = RandomForestClassifier(bootstrap=False,
                                   class_weight='balanced',
                                    criterion='log_loss',
                                   max_depth=20,
                                   min_samples_leaf=1,
                                   min_samples_split=2,
                                   n_estimators=250)
model_grid.fit(X_train, y_train)
y_pred_grid = model_grid.predict(X_test)
print(classification_report(y_pred_grid, y_test))

              precision    recall  f1-score   support

          no       0.80      0.80      0.80         5
         yes       0.67      0.67      0.67         3

    accuracy                           0.75         8
   macro avg       0.73      0.73      0.73         8
weighted avg       0.75      0.75      0.75         8



### XGboost

In [ ]:
dic = {'yes': 1, 'no': 0}
y_train = y_train.replace(dic)
y_test = y_test.replace(dic)
print(y_train.value_counts())

rejection
0    21
1     9
Name: count, dtype: int64


C:\Users\eyadh\AppData\Local\Temp\ipykernel_13424\1283535461.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace(dic)
C:\Users\eyadh\AppData\Local\Temp\ipykernel_13424\1283535461.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace(dic)


In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV

# Calculate scale_pos_weight to handle class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

param_grid_xgb = {
    'n_estimators': [100, 250],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'scale_pos_weight': [scale_pos_weight]
}

grid_search_xgb = GridSearchCV(XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'), 
                               param_grid=param_grid_xgb, 
                               cv=5, 
                               scoring='recall_weighted',
                               n_jobs=-1)

grid_search_xgb.fit(X_train, y_train)

print("Best Parameters:", grid_search_xgb.best_params_)
print("Best Score (Recall Weighted):", grid_search_xgb.best_score_)

c:\Users\eyadh\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:06:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 250, 'scale_pos_weight': 2.3333333333333335, 'subsample': 0.7}
Best Score (Recall Weighted): 0.8


In [ ]:
# Train XGBoost model with best parameters
model_xgb = XGBClassifier(
    colsample_bytree=0.7,
    learning_rate=0.01,
    max_depth=3,
    n_estimators=250,
    scale_pos_weight=2.3333333333333335,
    subsample=0.7,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model_xgb.fit(X_train, y_train)
y_pred_xgb = model_xgb.predict(X_test)

print("XGBoost Classification Report:")
print(classification_report(y_pred_xgb, y_test))

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.80      0.80         5
           1       0.67      0.67      0.67         3

    accuracy                           0.75         8
   macro avg       0.73      0.73      0.73         8
weighted avg       0.75      0.75      0.75         8



c:\Users\eyadh\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:06:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
import joblib


# Après avoir entraîné ton modèle
model_xgb.fit(X_train, y_train)

# Sauvegarder le modèle dans un fichier
joblib.dump(model_xgb, "model.pkl")
joblib.dump(transformer, "transformer.pkl")



c:\Users\eyadh\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:06:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


['transformer.pkl']

In [ ]:
joblib.dump(X_train, "X_train.pkl")
joblib.dump(y_train, "y_train.pkl")

['y_train.pkl']